In [0]:
%sh pwd

In [0]:
%run "/Workspace/Users/abhishekkumar.singh@techygeekhub.com/FoodQuest P&L/Actual Management P&L"

In [0]:
%run "/Workspace/Users/abhishekkumar.singh@techygeekhub.com/FoodQuest P&L/Flat Actual Management P&L"

In [0]:
%run "/Workspace/Users/abhishekkumar.singh@techygeekhub.com/FoodQuest P&L/Budget Management P&L"

In [0]:
df_actual_selected = df_final_actual.select('city', 'management_sort_order', 'location_id', 'store_type', 'major_group', 'mapped_name', 'Detail/Total', 'netsuite_location_name', 'account_name', 'zone', 'sub_group', 'management_details_total', 'type', 'year', 'account_type', 'store_open_date2', 'brand_id', 'company_id', 'month', 'parent_company', 'country_code', 'group', 'management_group', 'amount')

df_py_selected = df_final_py.select('city', 'management_sort_order', 'location_id', 'store_type', 'major_group', 'mapped_name', 'Detail/Total', 'netsuite_location_name', 'account_name', 'zone', 'sub_group', 'management_details_total', 'type', 'year', 'account_type', 'store_open_date2', 'brand_id', 'company_id', 'month', 'parent_company', 'country_code', 'group', 'management_group', 'py_amount').withColumn('year', col('year')+1)

df_budget_selected = df_final_budget.select('city', 'management_sort_order', 'location_id', 'store_type', 'major_group', 'mapped_name', 'Detail/Total', 'netsuite_location_name', 'account_name', 'zone', 'sub_group', 'management_details_total', 'type', 'year', 'account_type', 'store_open_date2', 'brand_id', 'company_id', 'month', 'parent_company', 'country_code', 'group', 'management_group', 'budget_amount')

In [0]:
df_union = df_actual_selected.unionAll(df_py_selected).unionAll(df_budget_selected)
df_union.filter(col('netsuite_location_name')=='ALB-100-Dubai Mall').display()

In [0]:
df_temp = df_actual_selected.join(df_py_selected, ['city', 'management_sort_order', 'location_id', 'store_type', 'major_group', 'mapped_name', 'Detail/Total', 'netsuite_location_name', 'account_name', 'zone', 'sub_group', 'management_details_total', 'type', 'year', 'account_type', 'store_open_date2', 'brand_id', 'company_id', 'month', 'parent_company', 'country_code', 'group', 'management_group'], 'outer')
# df_temp.display()
df_temp = df_temp.join(df_budget_selected, ['city', 'management_sort_order', 'location_id', 'store_type', 'major_group', 'mapped_name', 'Detail/Total', 'netsuite_location_name', 'account_name', 'zone', 'sub_group', 'management_details_total', 'type', 'year', 'account_type', 'store_open_date2', 'brand_id', 'company_id', 'month', 'parent_company', 'country_code', 'group', 'management_group'], 'outer')

In [0]:
from functools import reduce
from pyspark.sql.functions import col

join_cols = ['city', 'management_sort_order', 'location_id', 'store_type', 'major_group', 'mapped_name', 'Detail/Total', 'netsuite_location_name', 'account_name', 'zone', 'sub_group', 'management_details_total', 'type', 'year', 'account_type', 'store_open_date2', 'brand_id', 'company_id', 'month', 'parent_company', 'country_code', 'group', 'management_group']

# Equisafe join for actual and py
join_condition_1 = reduce(lambda a, b: a & b, [col(f'actual.{c}').eqNullSafe(col(f'py.{c}')) for c in join_cols])
df_actual_selected = df_actual_selected.alias('actual')
df_py_selected = df_py_selected.alias('py')
df_temp = df_actual_selected.join(df_py_selected, join_condition_1, 'outer')

# Equisafe join for temp and budget
df_budget_selected = df_budget_selected.alias('budget')
join_condition_2 = reduce(lambda a, b: a & b, [col(f'temp.{c}').eqNullSafe(col(f'budget.{c}')) for c in join_cols])
df_temp = df_temp.alias('temp').join(df_budget_selected, join_condition_2, 'outer')

In [0]:
from pyspark.sql import functions as F

join_keys = ['city', 'management_sort_order', 'location_id', 'store_type', 'major_group', 
             'mapped_name', 'Detail/Total', 'netsuite_location_name', 'account_name', 'zone', 
             'sub_group', 'management_details_total', 'type', 'year', 'account_type', 
             'store_open_date2', 'brand_id', 'company_id', 'month', 'parent_company', 
             'country_code', 'group', 'management_group']

# Get the non-key (metric) columns from each df
actual_metrics = [c for c in df_actual_selected.columns if c not in join_keys]
py_metrics     = [c for c in df_py_selected.columns if c not in join_keys]
budget_metrics = [c for c in df_budget_selected.columns if c not in join_keys]

# Add missing metric columns as null so all 3 dfs have the same schema
all_metrics = list(set(actual_metrics + py_metrics + budget_metrics))

def add_missing_cols(df, all_cols, existing_cols):
    for col in all_cols:
        if col not in existing_cols:
            df = df.withColumn(col, F.lit(None).cast("double"))  # adjust type if needed
    return df

df_actual_full = add_missing_cols(df_actual_selected, all_metrics, actual_metrics)
df_py_full     = add_missing_cols(df_py_selected,     all_metrics, py_metrics)
df_budget_full = add_missing_cols(df_budget_selected, all_metrics, budget_metrics)

# Union all three and group by keys, taking first non-null value per metric column
df_temp = (
    df_actual_full
    .unionByName(df_py_full)
    .unionByName(df_budget_full)
    .groupBy(join_keys)
    .agg(*[F.first(c, ignorenulls=True).alias(c) for c in all_metrics])
).orderBy(
            "parent_company", "company_id", "brand_id", 
            "netsuite_location_name", 'year', 'month', 'management_sort_order'
        )
# df_temp.display()

In [0]:
df_final = df_temp.filter(col('netsuite_location_name').isNotNull()).filter(col('year')==2026).filter(col('month')=='JAN')
df_final.display()